In [ ]:
#Link the pattern with data using this

In [1]:
import scipy.io as sio
import numpy as np
import os

# Configuration: {mat_filename: pattern_subfolder}
mat_files = {
    '0.mat': 'pattern_cir_0',
    '2pibi3.mat': 'pattern_cir_2pibi3',
    '4pibi3.mat': 'pattern_cir_4pibi3',
}

data_dir = 'data'
pattern_dir = 'pattern_360'

# Grid parameters
start = 101
step = 10

# Loop through each phase
for mat_name, pattern_folder in mat_files.items():
    phase = pattern_folder.split('_')[-1]  # Extract "0", "2pibi3", or "4pibi3"
    mat_path = os.path.join(data_dir, mat_name)

    print(f"Processing: {mat_name} ({phase})")

    mat_data = sio.loadmat(mat_path)
    key = next(k for k in mat_data.keys() if not k.startswith('__'))
    data = mat_data[key]
    h, w = data.shape
    # print(h, w)
    A_values = []
    pattern_paths = []

    for i, kx in enumerate(range(start, min(h+1,350), step)):
        for j, ky in enumerate(range(start, min(w+1,360), step)):
            val = data[kx-1, ky-1]
            if val != 0:
                filename = f"pat{phase}_kx_{kx}_ky_{ky}.bmp"
                relative_path = os.path.join(pattern_folder, filename)
                A_values.append(val)
                pattern_paths.append(relative_path)

    # Save per phase
    output_dict = {
        'A': np.array(A_values).reshape(-1, 1),
        'pattern_paths': np.array(pattern_paths, dtype=object)
    }

    save_name = f'linked_{phase}.mat'
    sio.savemat(save_name, output_dict)
    print(f"Saved: {save_name} ({len(A_values)} entries)")

Processing: 0.mat (0)
Saved: linked_0.mat (650 entries)
Processing: 2pibi3.mat (2pibi3)
Saved: linked_2pibi3.mat (650 entries)
Processing: 4pibi3.mat (4pibi3)
Saved: linked_4pibi3.mat (650 entries)


In [2]:
import scipy.io as sio
import numpy as np
from PIL import Image
import os

def load_phase_data(mat_path, pattern_root='pattern_360', resize_to=None):
    """
    Loads image patterns and associated intensity values from a linked_phaseX.mat file.

    Args:
        mat_path (str): Path to the .mat file (e.g., linked_phase0.mat)
        pattern_root (str): Root folder where pattern folders are located.
        resize_to (tuple): If provided, resize each image to this size (e.g., (64, 64))

    Returns:"
        images (np.ndarray): Array of shape (N, H, W) or (N, H, W, C)
        labels (np.ndarray): Array of shape (N, 1)
    """
    data = sio.loadmat(mat_path)
    A = data['A']  # (N, 1)
    pattern_paths = data['pattern_paths'].squeeze()  # (N,)

    images = []
    for path in pattern_paths:
        full_path = os.path.join(pattern_root, path[0] if isinstance(path, np.ndarray) else path)
        img = Image.open(full_path).convert('L')  # grayscale
        if resize_to:
            img = img.resize(resize_to)
        img_np = np.array(img)
        images.append(img_np)

    images = np.stack(images)  # (N, H, W)
    return images, A


In [3]:
import matplotlib.pyplot as plt

def visualize_patterns(images, labels, num=5):
    """
    Display a few pattern images with their corresponding intensity values.
    
    Args:
        images (np.ndarray): Array of pattern images (N, H, W)
        labels (np.ndarray): Array of intensity values (N, 1)
        num (int): Number of samples to display
    """
    num = min(num, len(images))  # Limit to available data
    plt.figure(figsize=(15, 3))

    for i in range(num):
        plt.subplot(1, num, i+1)
        plt.imshow(images[i], cmap='gray')
        plt.title(f"Intensity: {labels[i][0]:.4f}")
        plt.axis('off')
    
    plt.tight_layout()
    plt.show()


In [4]:
import scipy.io as sio

data = sio.loadmat('linked_0.mat')
# print(data)
A = data['A']         # (N, 1)
paths = data['pattern_paths'].squeeze()  # (N,)

# Print the first 10 entries
for i in range(10):
    path = paths[i][0] if isinstance(paths[i], np.ndarray) else paths[i]
    print(f"{i}: {path} → Intensity: {A[i][0]:.4f}")


0: pattern_cir_0/pat0_kx_101_ky_101.bmp → Intensity: 925.7593
1: pattern_cir_0/pat0_kx_101_ky_111.bmp → Intensity: 922.2531
2: pattern_cir_0/pat0_kx_101_ky_121.bmp → Intensity: 922.0111
3: pattern_cir_0/pat0_kx_101_ky_131.bmp → Intensity: 926.9158
4: pattern_cir_0/pat0_kx_101_ky_141.bmp → Intensity: 933.9388
5: pattern_cir_0/pat0_kx_101_ky_151.bmp → Intensity: 940.1720
6: pattern_cir_0/pat0_kx_101_ky_161.bmp → Intensity: 935.4003
7: pattern_cir_0/pat0_kx_101_ky_171.bmp → Intensity: 941.1964
8: pattern_cir_0/pat0_kx_101_ky_181.bmp → Intensity: 938.4833
9: pattern_cir_0/pat0_kx_101_ky_191.bmp → Intensity: 938.7002


In [6]:
def generate_linked_mat(data_path, variable_name, phase_tag, save_to):
    """
    Links non-zero values in the matrix to pattern paths and saves a .mat file.

    Args:
        data_path (str): Path to the input .mat file
        variable_name (str): Name of the matrix variable inside the .mat file
        phase_tag (str): Phase name for folder and filename generation
        save_to (str): Output path for the linked .mat file
    """
    mat = sio.loadmat(data_path)
    matrix = mat[variable_name]

    values, paths = [], []

    for i in range(101,min(matrix.shape[0]+1,350)):
        for j in range(101,min(matrix.shape[1]+1,360)):
            val = matrix[i-1, j-1]
            if val != 0:
                kx = i
                ky = j 
                values.append([val])
                filename = f"pattern_cir_{phase_tag}/pat{phase_tag[0]}_kx_{kx}_ky_{ky}.bmp"
                paths.append([filename])

    A = np.array(values)
    pattern_paths = np.array(paths)

    sio.savemat(save_to, {'A': A, 'pattern_paths': pattern_paths})
    print(f"✅ Saved {save_to} with {len(A)} patterns linked.")


In [7]:
generate_linked_mat('data/0.mat', 'Sum_II_0', '0', 'linked_0.mat')
generate_linked_mat('data/2pibi3.mat', 'Sum_II_2pibi3', '2pibi3', 'linked_2pibi3.mat')
generate_linked_mat('data/4pibi3.mat', 'Sum_II_4pibi3', '4pibi3', 'linked_4pibi3.mat')


✅ Saved linked_0.mat with 650 patterns linked.
✅ Saved linked_2pibi3.mat with 650 patterns linked.
✅ Saved linked_4pibi3.mat with 650 patterns linked.


In [16]:
mat_4 = sio.loadmat('data/4pibi3.mat')
print(mat_4.keys())


dict_keys(['__header__', '__version__', '__globals__', 'Sum_II_4pibi3'])


In [20]:
import scipy.io as sio

def preview_linked_data(mat_path, num=10):
    """
    Prints the first few pattern paths and their associated intensity values.

    Args:
        mat_path (str): Path to linked .mat file (e.g., 'linked_phase0.mat')
        num (int): Number of entries to preview
    """
    data = sio.loadmat(mat_path)
    A = data['A'].squeeze()  # shape (N,)
    paths = data['pattern_paths'].squeeze()  # shape (N,)

    print(f"🔍 Previewing {num} patterns from {mat_path}:\n")
    for i in range(min(num, len(A))):
        path = paths[i]
        if isinstance(path, np.ndarray):
            path = path[0]
        print(f"{i}: {path} → Intensity: {A[i]:.4f}")


In [22]:
preview_linked_data('linked_0.mat', num=10)
preview_linked_data('linked_2pibi3.mat', num=10)
preview_linked_data('linked_4pibi3.mat', num=10)


🔍 Previewing 10 patterns from linked_0.mat:

0: pattern_cir_0/pat0_kx_101_ky_101.bmp  → Intensity: 925.7593
1: pattern_cir_0/pat0_kx_101_ky_111.bmp  → Intensity: 922.2531
2: pattern_cir_0/pat0_kx_101_ky_121.bmp  → Intensity: 922.0111
3: pattern_cir_0/pat0_kx_101_ky_131.bmp  → Intensity: 926.9158
4: pattern_cir_0/pat0_kx_101_ky_141.bmp  → Intensity: 933.9388
5: pattern_cir_0/pat0_kx_101_ky_151.bmp  → Intensity: 940.1720
6: pattern_cir_0/pat0_kx_101_ky_161.bmp  → Intensity: 935.4003
7: pattern_cir_0/pat0_kx_101_ky_171.bmp  → Intensity: 941.1964
8: pattern_cir_0/pat0_kx_101_ky_181.bmp  → Intensity: 938.4833
9: pattern_cir_0/pat0_kx_101_ky_191.bmp  → Intensity: 938.7002
🔍 Previewing 10 patterns from linked_2pibi3.mat:

0: pattern_cir_2pibi3/pat2_kx_101_ky_101.bmp  → Intensity: 940.6831
1: pattern_cir_2pibi3/pat2_kx_101_ky_111.bmp  → Intensity: 942.9705
2: pattern_cir_2pibi3/pat2_kx_101_ky_121.bmp  → Intensity: 946.7740
3: pattern_cir_2pibi3/pat2_kx_101_ky_131.bmp  → Intensity: 949.7457
4: 

In [266]:
print(os.path.exists('pattern/pattern_cir_0/pat0_kx_101_ky_101.bmp'))  # should match exactly


True
